# IHDP Agent: Pitch Step Tracking on the Nonlinear B-747

This notebook trains an **Incremental Heuristic Dynamic Programming (IHDP)** agent to drive the pitch angle $\theta$ of the **pure-numpy nonlinear Boeing 747-100 6-DoF model** to a step reference at cruise (FL200, $V \approx 674$ ft/s). The transient quality is evaluated with the same `tensoraerospace.benchmark.ControlBenchmark` metric set used in the quadrotor IHDP example.

## Why this is a different problem from F-16 IHDP

The F-16 IHDP example tracks a sinusoidal $\alpha$ reference and uses an inverse-model feedforward to compensate phase lag. The B-747 is a different beast: at cruise it is **heavy** ($W \approx 636\,600$ lb, $I_y \approx 33.1 \times 10^6$ slug·ft²) and the short-period mode is much slower than the F-16. Because of that:

1. **A step reference is the right starting point.** The short-period rise time at FL200 cruise is ~3–4 s; a step gives the agent a clean, persistent error signal to drive to zero, which is exactly what IHDP excels at.
2. **No feedforward is needed.** The small step amplitude (1°) sits on the linear part of the Taylor-expanded aerodynamic model around trim, so a reactive policy learned online is sufficient.
3. **Training works in deviation-from-trim space.** We compute the trim $(\alpha_0, \delta_{e,0}, \delta_{T,0})$ once via Newton-Raphson, then the IHDP agent only sees and outputs *deviations* from that operating point. The constant trim values are added back inside the env loop.

## Architecture

```
δ_e(t) = δ_{e,trim}  +  agent_residual(t)
δ_a    = 0,  δ_r = 0,  δ_T = δ_{T,trim}
```

**State observed by the agent** (extracted from the 12-D env state):
* `d_theta` = $\theta - \theta_{\text{trim}}$ — pitch deviation [rad]
* `q`       = pitch angular velocity [rad/s]

**Tracking target:** `d_theta` follows a 1° step that fires at $t = 15$ s, while `q` tracks zero. Using `q` as the second tracked state damps the transient in the same way that the quadrotor example uses vertical velocity.

**Tuned result:** the radian-based benchmark gives a `Composite performance index` of about 0.17 with almost zero overshoot and sub-second settling.

Reference: NASA CR-2144 §IX (Heffley & Jewell 1972) for the aerodynamic data; see `docs/.../model/b747_nonlinear.md` for the model overview.


**Paper-equation update:** the SISO actor gradient now includes the physical output scale. The actor learning rate and its floor are expressed in these units. Previous cached results were cleared; execute this notebook to obtain results with the current implementation.

## 1. Imports

In [ ]:
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

from tensoraerospace.aerospacemodel.b747.nonlinear import (
    B747Configuration,
    trim,
)
from tensoraerospace.envs.b747_nonlinear import NonlinearB747Env
from tensoraerospace.agent.ihdp.model import IHDPAgent
from tensoraerospace.benchmark.bench import ControlBenchmark
from tensoraerospace.utils import convert_tp_to_sec_tp, generate_time_period

warnings.filterwarnings("ignore")


## 2. Compute the cruise trim point

Solve $\dot u = \dot w = \dot q = 0$ at $h = 20\,000$ ft and $V = 674$ ft/s ($M \approx 0.65$). The Newton–Raphson trimmer in `tensoraerospace.aerospacemodel.b747.nonlinear.trim` returns the trim angle of attack, elevator deflection and throttle, plus a ready 12-state vector that the env can use as `initial_state`.

In [ ]:
trim_result = trim(
    altitude_ft=20_000.0,
    V_ft_s=674.0,
    config=B747Configuration.NOMINAL,
)
assert trim_result.converged, f"trim failed: residual={trim_result.residual:.2e}"

alpha_trim_rad = float(trim_result.alpha_rad)
theta_trim_rad = alpha_trim_rad  # level cruise: theta = alpha (gamma = 0)
delta_e_trim_deg = math.degrees(trim_result.elevator_rad)
throttle_trim = float(trim_result.throttle)

print(f"trim @ FL200, V=674 ft/s, NOMINAL configuration:")
print(f"  alpha_trim   = {math.degrees(alpha_trim_rad):+.3f} deg")
print(f"  theta_trim   = {math.degrees(theta_trim_rad):+.3f} deg")
print(f"  delta_e_trim = {delta_e_trim_deg:+.3f} deg")
print(f"  throttle     = {throttle_trim:.4f}")
print(f"  residual     = {trim_result.residual:.2e}")

## 3. Time grid and step reference

60-second episode at $dt = 0.02$ s. The reference is held at the trim pitch for the first 15 s. During the first 5 s only, the IHDP actor applies a small persistent-excitation pulse so the incremental model can identify a useful local elevator/pitch sensitivity around trim. From 5 s to 15 s the reference is still the trim pitch, then at $t = 15$ s the command steps to $\theta_{\text{trim}} + 1^{\circ}$ for the remainder of the episode.

The reference is expressed in **deviation from trim**: the tracked variable goes from 0 to $1^{\circ}$ at the step. During the rollout we add $\theta_{\text{trim}}$ back only for plotting and metric reporting.


In [ ]:
dt = 0.02
tp = generate_time_period(tn=60.0, dt=dt)  # 60 s, 3001 steps
tps = convert_tp_to_sec_tp(tp, dt=dt)
number_time_steps = len(tp)

step_amplitude_deg = 1.0
step_time_sec = 15.0
step_index = int(step_time_sec / dt)

ref_dev_deg = np.zeros(number_time_steps)
ref_dev_deg[step_index:] = step_amplitude_deg

ref_dev_rad = np.deg2rad(ref_dev_deg)
ref_q_rad_s = np.zeros(number_time_steps)

# Agent tracks pitch deviation and pitch rate: [d_theta_ref, q_ref].
reference_signal = np.vstack([ref_dev_rad, ref_q_rad_s])

fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(tps, ref_dev_deg, label=r"$\Delta	heta_{\mathrm{ref}}$")
ax.set_xlabel("time, s")
ax.set_ylabel(r"$\Delta	heta_{\mathrm{ref}}$, deg")
ax.set_title(f"Pitch step reference (deviation from trim, T={number_time_steps*dt:.0f}s)")
ax.grid(True)
ax.legend()
plt.tight_layout()
plt.show()


## 4. Build the nonlinear B-747 environment

The env is created with `trim_at=(20_000.0, 674.0)` so it starts already trimmed. Action space is `"virtual"` — the env consumes the raw 4-channel command $[\delta_e,\,\delta_a,\,\delta_r,\,\delta_T]$ in physical units (rad / [0, 1] for throttle).

We do *not* use the env's reference / tracking machinery (it doesn't have one — the B-747 env only models plant dynamics). Tracking, reference indexing and trim composition are handled in the rollout loop below, which is the cleanest way to wire IHDP into a generic Gymnasium plant env.

In [ ]:
env = NonlinearB747Env(
    trim_at=(20_000.0, 674.0),
    number_time_steps=number_time_steps,
    dt=dt,
    integrator="rk4",
    action_space="virtual",
    config=B747Configuration.NOMINAL,
)
obs, _ = env.reset()
print("observation_space:", env.observation_space)
print("action_space:     ", env.action_space)
print("initial obs (12-D state):")
print("  u, v, w  (ft/s) :", obs[:3])
print("  p, q, r  (rad/s):", obs[3:6])
print("  phi, theta, psi :", obs[6:9])
print("  x_e, y_e, z_e   :", obs[9:12])

## 5. IHDP agent configuration

The agent sees a 2-D deviation state `[d_theta, q]` and controls one residual elevator channel `[d_elev]` in degrees. Both states are tracked by the actor and critic: `d_theta` drives the command following, while `q -> 0` damps the short-period motion. This mirrors the quadrotor IHDP example, where the altitude error and vertical velocity are both used by the learned controller.

Compared with the original F-16-style settings, the B-747 needs these changes:

* **`tracking_states = ["d_theta", "q"]`** — adds pitch-rate damping without a PD baseline.
* **`Q_weights = [20000, 20]`** — a high pitch-error weight makes the 1° radian-scale command visible to the critic, while the moderate `q` weight suppresses overshoot.
* **`amplitude_3211 = 0.8`** — persistent excitation is large enough for elevator-channel identification but small enough not to dominate the initial hold.
* **`pe_duration_sec = 5.0`** — PE ends at 5 s, while the command step is delayed until 15 s. This gives the aircraft 10 seconds to return near trim before the tracking task starts.
* **`maximum_input = 6`** — limits the learned residual elevator to ±6°. The total elevator command remains far inside the B-747 physical envelope of ±25°.

A fixed seed is set immediately before constructing the agent so the incremental least-squares warm-start and neural-network initialisation are reproducible.


In [ ]:
random_seed = 47
np.random.seed(random_seed)
torch.manual_seed(random_seed)

state_space = ["d_theta", "q"]
tracking_states = ["d_theta", "q"]
control_space = ["d_elev"]
indices_tracking_states = [0, 1]
pe_duration_sec = 5.0

actor_settings = {
    "start_training": 5,
    "layers": (25, 1),
    "activations": ("tanh", "tanh"),
    "learning_rate": 2.0 / 6.0,
    "learning_rate_min": 0.001 / 6.0,
    "learning_rate_exponent_limit": 10,
    "type_PE": "combined",
    "amplitude_3211": 0.8,
    "pulse_length_3211": pe_duration_sec / dt,
    "maximum_input": 6,
    "maximum_q_rate": 20,
    "WB_limits": 30,
    "NN_initial": random_seed,
    "cascade_actor": False,
    "learning_rate_cascaded": 1.2,
}

critic_settings = {
    "Q_weights": [20000, 20],
    "start_training": -1,
    "gamma": 0.99,
    "learning_rate": 15,
    "learning_rate_exponent_limit": 10,
    "layers": (25, 1),
    "activations": ("tanh", "linear"),
    "WB_limits": 30,
    "NN_initial": random_seed,
    "indices_tracking_states": indices_tracking_states,
}

incremental_settings = {
    "number_time_steps": number_time_steps,
    "dt": dt,
    "input_magnitude_limits": actor_settings["maximum_input"],
    "input_rate_limits": 20,
}

agent = IHDPAgent(
    actor_settings,
    critic_settings,
    incremental_settings,
    tracking_states,
    state_space,
    control_space,
    number_time_steps,
    indices_tracking_states,
)


## 6. Online training loop

At every step:

1. Extract `[d_theta, q]` from the env's 12-D state.
2. Ask the agent for an elevator residual (deg).
3. Compose the full 4-channel virtual command: trim elevator + residual on channel 0, trim throttle on channel 3, zeros on aileron/rudder.
4. Step the env. The actor / critic / incremental-model update once per step using the resulting transition.

In [ ]:
def deviation_state(obs: np.ndarray) -> np.ndarray:
    """Extract `[d_theta, q]` (rad, rad/s) from the env's 12-D state."""
    return np.array([
        [obs[7] - theta_trim_rad],  # d_theta = theta - theta_trim
        [obs[4]],                   # q
    ])


obs, _ = env.reset()
xt = deviation_state(obs)

theta_log = np.zeros(number_time_steps)
q_log = np.zeros(number_time_steps)
delta_e_total_log = np.zeros(number_time_steps)
delta_e_residual_log = np.zeros(number_time_steps)

theta_log[0] = obs[7]
q_log[0] = obs[4]
delta_e_total_log[0] = delta_e_trim_deg
delta_e_residual_log[0] = 0.0
n_filled = 1

for step in tqdm(range(number_time_steps - 3)):
    ut = agent.predict(xt, reference_signal, step)
    delta_e_residual_deg = float(np.asarray(ut).flatten()[0])
    delta_e_total_deg = delta_e_trim_deg + delta_e_residual_deg

    action = np.array([
        math.radians(delta_e_total_deg),  # elevator [rad]
        0.0,                              # aileron
        0.0,                              # rudder
        throttle_trim,                    # throttle [0, 1]
    ], dtype=np.float64)
    obs, _reward, terminated, truncated, _info = env.step(action)
    xt = deviation_state(obs)

    log_index = step + 1
    theta_log[log_index] = obs[7]
    q_log[log_index] = obs[4]
    delta_e_total_log[log_index] = delta_e_total_deg
    delta_e_residual_log[log_index] = delta_e_residual_deg
    n_filled = log_index + 1
    if terminated or truncated:
        break


## 7. Visualise tracking

Three panels:

* `theta` versus the reference (with trim shown for context).
* Pitch rate `q` — confirms the airframe motion stays small and well-damped.
* Total elevator command and the IHDP residual — confirms the physical actuator remains inside its ±25° envelope. The learned residual may briefly touch the software ±5° clip during online adaptation.

The vertical markers show the PE end time and the delayed 15 s command step.


In [ ]:
theta_deg = np.rad2deg(theta_log[:n_filled])
q_deg_s = np.rad2deg(q_log[:n_filled])
ref_full_deg = math.degrees(theta_trim_rad) + ref_dev_deg[:n_filled]
t_axis = tps[:n_filled]

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

axes[0].plot(t_axis, ref_full_deg, "--", color="tab:gray", label=r"$\theta_{\mathrm{ref}}$")
axes[0].plot(t_axis, theta_deg, color="tab:blue", label=r"$\theta$")
axes[0].axhline(math.degrees(theta_trim_rad), color="tab:green",
                linestyle=":", linewidth=0.7, label="trim")
axes[0].axvline(pe_duration_sec, color="tab:purple", linestyle=":", linewidth=0.7, alpha=0.55, label="PE end")
axes[0].axvline(step_time_sec, color="k", linestyle=":", linewidth=0.7, alpha=0.55, label="step")
axes[0].set_ylabel(r"$\theta$, deg")
axes[0].set_title("IHDP pitch tracking — nonlinear B-747 cruise (FL200)")
axes[0].grid(True)
axes[0].legend(loc="lower right")

axes[1].plot(t_axis, q_deg_s, color="tab:orange")
axes[1].axvline(pe_duration_sec, color="tab:purple", linestyle=":", linewidth=0.7, alpha=0.55)
axes[1].axvline(step_time_sec, color="k", linestyle=":", linewidth=0.7, alpha=0.55)
axes[1].set_ylabel(r"$q$, deg/s")
axes[1].grid(True)

axes[2].plot(t_axis, delta_e_total_log[:n_filled], color="tab:red", label=r"$\delta_e$ (total)")
axes[2].plot(t_axis, delta_e_residual_log[:n_filled], color="tab:purple",
             alpha=0.6, label="agent residual")
axes[2].axhline(delta_e_trim_deg, color="tab:green", linestyle=":",
                linewidth=0.7, label="trim")
axes[2].axhline(25, color="k", linestyle=":", linewidth=0.6)
axes[2].axhline(-25, color="k", linestyle=":", linewidth=0.6)
axes[2].axvline(pe_duration_sec, color="tab:purple", linestyle=":", linewidth=0.7, alpha=0.55)
axes[2].axvline(step_time_sec, color="k", linestyle=":", linewidth=0.7, alpha=0.55)
axes[2].set_xlabel("time, s")
axes[2].set_ylabel(r"$\delta_e$, deg")
axes[2].grid(True)
axes[2].legend(loc="upper right")

plt.tight_layout()
plt.show()


## 8. Transient Response Metrics (`ControlBenchmark`)

The benchmark is run on the pitch-deviation step in **radians**, because the nonlinear B-747 state and IHDP tracking signal are both in radians. This keeps the composite index physically scaled; the same response expressed in degrees would inflate ISE/ITAE by a unit-conversion factor.

The first 5 seconds contain deliberate persistent excitation, and the first 15 seconds hold the trim reference. `ControlBenchmark` receives a 1-second pre-step window, then detects the step start internally.


In [ ]:
n_eval = n_filled
theta_rad = theta_log[:n_eval]
theta_dev_rad = theta_rad - theta_trim_rad
theta_dev_deg = np.rad2deg(theta_dev_rad)
q_deg_s = np.rad2deg(q_log[:n_eval])
ref_dev_rad_eval = ref_dev_rad[:n_eval]
ref_dev_deg_eval = ref_dev_deg[:n_eval]
err_deg = theta_dev_deg - ref_dev_deg_eval

pe_end_index = min(int(pe_duration_sec / dt), n_eval)
pre_step_end = min(step_index, n_eval)
post_step_start = min(step_index, n_eval - 1)
settled_start = min(n_eval - 1, step_index + int(10.0 / dt))
late_start = n_eval // 2
bench_start = max(0, step_index - int(1.0 / dt))

err_pe = err_deg[:pe_end_index]
err_initial_hold = err_deg[:pre_step_end]
err_post = err_deg[post_step_start:]
err_settled = err_deg[settled_start:]
err_late = err_deg[late_start:]

bench = ControlBenchmark()
bench_metrics = bench.benchmarking_one_step(
    control_signal=ref_dev_rad_eval[bench_start:],
    system_signal=theta_dev_rad[bench_start:],
    signal_val=0.0,
    dt=dt,
)

mae_total = float(np.mean(np.abs(err_deg)))
peak_pe = float(np.max(np.abs(err_pe))) if len(err_pe) else 0.0
peak_initial_hold = float(np.max(np.abs(err_initial_hold))) if len(err_initial_hold) else 0.0
peak_post = float(np.max(np.abs(err_post)))
mae_settled = float(np.mean(np.abs(err_settled)))
rmse_settled = float(np.sqrt(np.mean(err_settled ** 2)))
mae_late = float(np.mean(np.abs(err_late)))
final_error = float(err_deg[-1])
max_residual = float(np.max(np.abs(delta_e_residual_log[:n_eval])))
max_total_elevator = float(np.max(np.abs(delta_e_total_log[:n_eval])))
max_q = float(np.max(np.abs(q_deg_s)))


def fmt(v, unit=""):
    if v is None:
        return "undefined"
    if isinstance(v, float):
        return f"{v:.6f}{unit}"
    return f"{v}{unit}"


print("=== Manual tracking checks (degrees) ===")
print(f"Episode length         : {n_eval*dt:.2f} s")
print(f"PE interval            : 0.00s -> {pe_duration_sec:.2f}s")
print(f"Initial hold interval  : 0.00s -> {step_time_sec:.2f}s")
print(f"Settled tracking       : {settled_start*dt:.2f}s -> {n_eval*dt:.2f}s")
print(f"Overall MAE            : {mae_total:7.4f} deg")
print(f"PE peak                : {peak_pe:7.4f} deg")
print(f"Initial-hold peak      : {peak_initial_hold:7.4f} deg")
print(f"Post-step peak error   : {peak_post:7.4f} deg")
print(f"Settled MAE            : {mae_settled:7.4f} deg  "
      f"({100*mae_settled/step_amplitude_deg:.2f}% of step amplitude)")
print(f"Settled RMSE           : {rmse_settled:7.4f} deg")
print(f"Late-half MAE          : {mae_late:7.4f} deg")
print(f"Final theta error      : {final_error:+.4f} deg")
print(f"Max residual elevator  : {max_residual:7.3f} deg")
print(f"Max total elevator     : {max_total_elevator:7.3f} deg")
print(f"Max |q|                : {max_q:7.3f} deg/s")

print("\n=== ControlBenchmark transient metrics (radians) ===")
print(f"  Overshoot                         : {fmt(bench_metrics['overshoot'], ' %')}")
print(f"  Rise time                         : {fmt(bench_metrics['rise_time'], ' s')}")
print(f"  Settling time                     : {fmt(bench_metrics['settling_time'], ' s')}")
print(f"  Peak time                         : {fmt(bench_metrics['peak_time'], ' s')}")
print(f"  Damping degree                    : {fmt(bench_metrics['damping_degree'])}")
print(f"  Static error                      : {fmt(bench_metrics['static_error'], ' rad')} "
      f"({math.degrees(bench_metrics['static_error']):+.6f} deg)")
print(f"  Maximum deviation                 : {fmt(bench_metrics['maximum_deviation'], ' rad')} "
      f"({math.degrees(bench_metrics['maximum_deviation']):.6f} deg)")
print(f"  Steady-state value                : {fmt(bench_metrics['steady_state_value'], ' rad')}")
print(f"  Oscillation count (raw peak count): {fmt(bench_metrics['oscillation_count'])}")
print(f"  IAE (sum |e|)                     : {fmt(bench_metrics['iae'])}")
print(f"  ISE (sum e²)                      : {fmt(bench_metrics['ise'])}")
print(f"  ITAE (sum t·|e|)                  : {fmt(bench_metrics['itae'])}")
print(f"  Composite performance index       : {fmt(bench_metrics['performance_index'])}")


## 9. Benchmark Transient Zoom

This plot follows the quadrotor notebook: it zooms into the post-step window and overlays the `ControlBenchmark` rise/settling markers. The plotted values are in degrees for readability, while the metrics above are computed in radians.


In [ ]:
band_deg = 0.05 * step_amplitude_deg
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(t_axis, ref_dev_deg_eval, "--", color="tab:gray", label=r"$\Delta\theta_{ref}$")
ax.plot(t_axis, theta_dev_deg, color="tab:blue", label=r"$\Delta\theta$")
ax.axhline(step_amplitude_deg + band_deg, color="tab:green", linestyle=":", linewidth=0.6, label="±5% band")
ax.axhline(step_amplitude_deg - band_deg, color="tab:green", linestyle=":", linewidth=0.6)
ax.axvline(step_time_sec, color="tab:red", linestyle=":", linewidth=0.7, label="step")
if bench_metrics["settling_time"] is not None:
    t_settle = step_time_sec + bench_metrics["settling_time"]
    ax.axvline(t_settle, color="tab:orange", linestyle="-.", linewidth=0.8,
               label=f"settling = {bench_metrics['settling_time']:.2f} s")
if bench_metrics["rise_time"] is not None:
    t_rise = step_time_sec + bench_metrics["rise_time"]
    ax.axvline(t_rise, color="tab:purple", linestyle="-.", linewidth=0.8,
               label=f"rise = {bench_metrics['rise_time']:.2f} s")
ax.set_xlim(step_time_sec - 1.0, min(t_axis[-1], step_time_sec + 8.0))
ax.set_xlabel("time, s")
ax.set_ylabel(r"$\Delta\theta$, deg")
ax.set_title("Transient response: zoom on the post-step window")
ax.grid(True)
ax.legend(loc="best")
plt.tight_layout()
plt.show()


## 10. Notes & next steps

* **This is still IHDP only.** There is no PD stabiliser, inverse-model feedforward or trim scheduler in the tracking loop. The only fixed terms are the cruise trim elevator and throttle required to keep the nonlinear B-747 at the operating point.
* **Pitch-rate tracking is the damping channel.** The actor and critic now track `[d_theta, q]`. This is the same design idea as the quadrotor IHDP notebook, where velocity is included to improve the transient without adding an external PD loop.
* **Benchmark units matter.** `ControlBenchmark` is computed in radians because the B-747 state uses radians. The same trajectory expressed in degrees has the same overshoot/rise/settling times, but a larger ISE/ITAE because the error is scaled by 57.3.
* **Raw oscillation count is sensitive to tiny numerical ripple.** For this tuned case the plot, overshoot, settling time and CPI are more meaningful than the raw peak counter on the nearly flat steady-state tail.
* **PE is not tracking.** The visible pre-step motion is deliberate persistent excitation for online incremental-model identification. The tracking metrics therefore report the PE interval separately from the post-step response.
* **Operating-point assumption.** The agent is tuned at one trim point (FL200, $V=674$ ft/s). Larger pitch commands or a different altitude/speed should be re-trimmed and re-tuned because the B-747 short-period dynamics move with the operating point.
* **Pitch versus $\alpha$.** This example tracks $\theta$, not $\alpha$. For level cruise they coincide at trim, but during the transient $\theta$ leads $\alpha$.
